# Biohub Video and Overlay Viewer

Use this notebook in Colab to inspect Biohub `.zarr` videos as 2D slices, z-projections, time sliders, GT overlays, and optional prediction/submission overlays.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount note:', repr(e))

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
BASE_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'reports'
SUPPORT_WHEELS = PROJECT_DIR / 'artifacts/biohub-tracking-support-pack-50ep-v1/wheels'

PRED_SUBMISSION_CSV = REPORT_DIR / 'deepcenter_local_validation_submission.csv'
CHECKPOINT_DIR = REPORT_DIR / 'deepcenter_full199_prediction_geff_checkpoint'

print('PROJECT_DIR:', PROJECT_DIR, PROJECT_DIR.exists())
print('BASE_DIR:', BASE_DIR, BASE_DIR.exists())
print('PRED_SUBMISSION_CSV:', PRED_SUBMISSION_CSV, PRED_SUBMISSION_CSV.exists())
print('CHECKPOINT_DIR:', CHECKPOINT_DIR, CHECKPOINT_DIR.exists())

def ensure_import(package_name, pip_name=None):
    try:
        __import__(package_name)
        return
    except Exception:
        pass
    pip_name = pip_name or package_name
    if SUPPORT_WHEELS.exists():
        cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--find-links', str(SUPPORT_WHEELS), pip_name]
    else:
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
    print('install:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

ensure_import('zarr')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
from IPython.display import HTML, display

try:
    from ipywidgets import interact, IntSlider, Dropdown, Checkbox, FloatSlider
    HAS_WIDGETS = True
except Exception as e:
    HAS_WIDGETS = False
    print('ipywidgets unavailable:', repr(e))

assert BASE_DIR.exists(), 'BASE_DIR missing. Mount Drive or fix PROJECT_DIR.'

## 2. Sample Index and Helpers

In [ ]:
def sample_ids(split='train'):
    return sorted(p.name[:-5] for p in (BASE_DIR / split).glob('*.zarr'))

TRAIN_IDS = sample_ids('train')
TEST_IDS = sample_ids('test')

print('train zarr:', len(TRAIN_IDS))
print('test zarr:', len(TEST_IDS))
print('train examples:', TRAIN_IDS[:5])
print('test examples:', TEST_IDS[:5])

def image_path(sample_id, split='train'):
    path = BASE_DIR / split / f'{sample_id}.zarr'
    if not path.exists() and split == 'train':
        alt = BASE_DIR / 'test' / f'{sample_id}.zarr'
        if alt.exists():
            return alt
    return path

def open_image(sample_id, split='train'):
    path = image_path(sample_id, split)
    arr = zarr.open(str(path / '0'), mode='r')
    return arr

def read_geff_nodes_direct(geff_path):
    geff_path = Path(geff_path)
    if not geff_path.exists():
        return pd.DataFrame(columns=['node_id', 't', 'z', 'y', 'x'])
    root = zarr.open(str(geff_path), mode='r')
    ids = np.asarray(root['nodes/ids'])
    data = {'node_id': ids}
    for key in ['t', 'z', 'y', 'x']:
        data[key] = np.asarray(root[f'nodes/props/{key}/values'])
    return pd.DataFrame(data).astype({'t': int, 'z': int, 'y': int, 'x': int})

def gt_nodes(sample_id):
    return read_geff_nodes_direct(BASE_DIR / 'train' / f'{sample_id}.geff')

submission_df = None
def load_submission_nodes():
    global submission_df
    if submission_df is None:
        if PRED_SUBMISSION_CSV.exists():
            submission_df = pd.read_csv(PRED_SUBMISSION_CSV)
            print('loaded submission:', PRED_SUBMISSION_CSV, submission_df.shape)
        else:
            submission_df = pd.DataFrame(columns=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x'])
            print('No submission CSV found:', PRED_SUBMISSION_CSV)
    return submission_df

def pred_nodes_from_submission(sample_id):
    df = load_submission_nodes()
    if df.empty:
        return pd.DataFrame(columns=['node_id', 't', 'z', 'y', 'x'])
    out = df[(df['dataset'].astype(str) == sample_id) & (df['row_type'] == 'node')][['node_id', 't', 'z', 'y', 'x']].copy()
    return out.astype({'t': int, 'z': int, 'y': int, 'x': int})

def pred_nodes_from_checkpoint(sample_id):
    path = CHECKPOINT_DIR / f'{sample_id}.geff'
    try:
        return read_geff_nodes_direct(path)
    except Exception as e:
        print('checkpoint direct read failed:', repr(e))
        return pd.DataFrame(columns=['node_id', 't', 'z', 'y', 'x'])

def robust_limits(img, low=1, high=99.8):
    vmin, vmax = np.percentile(img, [low, high])
    if vmax <= vmin:
        vmax = float(np.max(img)) if np.max(img) > vmin else vmin + 1
    return vmin, vmax

## 3. Single Slice / Slab Overlay

In [ ]:
def plot_frame(
    sample_id='6bba_09961292',
    split='train',
    t=9,
    z=9,
    projection='slice',
    slab=1,
    show_gt=True,
    show_pred=False,
    pred_source='submission',
    figsize=7,
):
    arr = open_image(sample_id, split)
    t = int(np.clip(t, 0, arr.shape[0] - 1))
    z = int(np.clip(z, 0, arr.shape[1] - 1))

    if projection == 'mip_full':
        img = np.asarray(arr[t]).max(axis=0)
        z_filter = None
        title_proj = 'full-z MIP'
    elif projection == 'mip_slab':
        z0 = max(0, z - slab)
        z1 = min(arr.shape[1], z + slab + 1)
        img = np.asarray(arr[t, z0:z1]).max(axis=0)
        z_filter = (z0, z1 - 1)
        title_proj = f'z-slab MIP {z0}..{z1 - 1}'
    else:
        img = np.asarray(arr[t, z])
        z_filter = (z - slab, z + slab)
        title_proj = f'z={z} slice +/- {slab}'

    vmin, vmax = robust_limits(img)
    plt.figure(figsize=(figsize, figsize))
    plt.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)

    if show_gt:
        g = gt_nodes(sample_id)
        g = g[g['t'] == t]
        if z_filter is not None:
            g = g[(g['z'] >= z_filter[0]) & (g['z'] <= z_filter[1])]
        if len(g):
            plt.scatter(g['x'], g['y'], s=42, facecolors='none', edgecolors='lime', linewidths=1.4, label=f'GT {len(g)}')

    if show_pred:
        if pred_source == 'checkpoint':
            p = pred_nodes_from_checkpoint(sample_id)
        else:
            p = pred_nodes_from_submission(sample_id)
        p = p[p['t'] == t]
        if z_filter is not None:
            p = p[(p['z'] >= z_filter[0]) & (p['z'] <= z_filter[1])]
        if len(p):
            plt.scatter(p['x'], p['y'], s=12, facecolors='none', edgecolors='red', linewidths=0.8, alpha=0.65, label=f'Pred {len(p)}')

    plt.title(f'{sample_id} | t={t} | {title_proj}')
    plt.axis('off')
    if show_gt or show_pred:
        plt.legend(loc='upper right')
    plt.show()

plot_frame(sample_id='6bba_09961292', split='train', t=9, z=9, projection='slice', slab=1, show_gt=True)

## 4. Interactive Viewer

In [ ]:
if HAS_WIDGETS:
    default_samples = TRAIN_IDS[:]
    preferred = ['6bba_09961292', '6bba_57b7cc1e', '6bba_afb141ff', '44b6_0c582fdc', '44b6_d754aa59']
    ordered = [s for s in preferred if s in default_samples] + [s for s in default_samples if s not in preferred]

    def interactive_plot(sample_id, t, z, projection, slab, show_gt, show_pred, pred_source):
        plot_frame(
            sample_id=sample_id,
            split='train',
            t=t,
            z=z,
            projection=projection,
            slab=slab,
            show_gt=show_gt,
            show_pred=show_pred,
            pred_source=pred_source,
            figsize=7,
        )

    interact(
        interactive_plot,
        sample_id=Dropdown(options=ordered, value=ordered[0], description='sample'),
        t=IntSlider(min=0, max=99, step=1, value=9, description='t'),
        z=IntSlider(min=0, max=63, step=1, value=32, description='z'),
        projection=Dropdown(options=['slice', 'mip_slab', 'mip_full'], value='slice', description='view'),
        slab=IntSlider(min=0, max=8, step=1, value=1, description='slab'),
        show_gt=Checkbox(value=True, description='GT'),
        show_pred=Checkbox(value=False, description='Pred'),
        pred_source=Dropdown(options=['submission', 'checkpoint'], value='submission', description='pred'),
    )
else:
    print('ipywidgets is not available. Use plot_frame(...) manually.')

## 5. Z Montage at One Timepoint

In [ ]:
def z_montage(sample_id='6bba_09961292', split='train', t=9, z_values=None, cols=4):
    arr = open_image(sample_id, split)
    if z_values is None:
        z_values = list(range(0, arr.shape[1], 8))
    rows = int(np.ceil(len(z_values) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.asarray(axes).reshape(-1)
    for ax, zz in zip(axes, z_values):
        img = np.asarray(arr[t, zz])
        vmin, vmax = robust_limits(img)
        ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_title(f'z={zz}')
        ax.axis('off')
    for ax in axes[len(z_values):]:
        ax.axis('off')
    fig.suptitle(f'{sample_id} | t={t} | z montage')
    plt.tight_layout()
    plt.show()

z_montage(sample_id='6bba_09961292', t=9)

## 6. Time Animation / GIF

In [ ]:
from matplotlib.animation import FuncAnimation

def make_mip_animation(sample_id='6bba_09961292', split='train', every=2, fps=8, save_gif=False):
    arr = open_image(sample_id, split)
    frames = list(range(0, arr.shape[0], every))
    first = np.asarray(arr[frames[0]]).max(axis=0)
    vmin, vmax = robust_limits(first)

    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(first, cmap='gray', vmin=vmin, vmax=vmax)
    title = ax.set_title(f'{sample_id} | t={frames[0]} | MIP')
    ax.axis('off')

    def update(i):
        tt = frames[i]
        img = np.asarray(arr[tt]).max(axis=0)
        im.set_data(img)
        title.set_text(f'{sample_id} | t={tt} | MIP')
        return im, title

    anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=False)
    plt.close(fig)

    if save_gif:
        out_path = REPORT_DIR / f'{sample_id}_mip_every{every}.gif'
        anim.save(out_path, writer='pillow', fps=fps)
        print('saved:', out_path)

    return HTML(anim.to_jshtml())

make_mip_animation(sample_id='6bba_09961292', every=4, fps=8, save_gif=False)

## 7. Quick Visual Spot-Check Grid

In [ ]:
spot_samples = [s for s in ['44b6_0c582fdc', '44b6_d754aa59', '6bba_afb141ff', '6bba_09961292', '6bba_57b7cc1e', '6bba_cdcfe533'] if s in TRAIN_IDS]

def spot_check_grid(samples=spot_samples, t=50):
    cols = 3
    rows = int(np.ceil(len(samples) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.asarray(axes).reshape(-1)
    for ax, sample_id in zip(axes, samples):
        arr = open_image(sample_id, 'train')
        tt = int(np.clip(t, 0, arr.shape[0] - 1))
        img = np.asarray(arr[tt]).max(axis=0)
        vmin, vmax = robust_limits(img)
        ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
        g = gt_nodes(sample_id)
        g = g[g['t'] == tt]
        if len(g):
            ax.scatter(g['x'], g['y'], s=28, facecolors='none', edgecolors='lime', linewidths=1.0)
        ax.set_title(f'{sample_id} | t={tt} | GT={len(g)}')
        ax.axis('off')
    for ax in axes[len(samples):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

spot_check_grid(t=50)